# H3 — Multi-chain liquidity providers: what the data supports

The LP tooling is in `crosschain_arb.lp` (`summarise_lp_events` → `build_lvr_panel` → `normalise_lvr` → `cohort_tests`, `multi_chain_ols`, `strict_negative_tail_fisher`) and is unit-tested on synthetic data. This notebook examines the two LP data sets in the repository: a per-wallet table (`lvr_table.parquet`) and per-chain liquidity ledgers. The numbers below are produced by `crosschain_arb.checks.lp_data_checks` (also saved by `scripts/run_data_checks.py`).

In [1]:
import pandas as pd
from IPython.display import Image, display

pd.options.display.float_format = "{:,.4f}".format


def show(path):
    display(Image(filename=str(path)))

In [2]:
from crosschain_arb import checks
from crosschain_arb.io import data_path

lvr_table = pd.read_parquet(data_path("lvr_table.parquet"))
led_eth = pd.read_parquet(data_path("ledger_ethereum_long.parquet"))
led_arb = pd.read_parquet(data_path("ledger_arbitrum_long.parquet"))
lp = checks.lp_data_checks(lvr_table, led_eth, led_arb)
ledgers = lp.pop("ledgers")

## The table's loss column

In [3]:
pd.Series(
    {
        k: lp[k]
        for k in [
            "table_rows",
            "table_unique_wallets",
            "table_median_abs_lvr",
            "table_rows_abs_lvr_below_1e4",
            "table_rows_abs_lvr_below_1e4_exactly_zero",
        ]
    }
)

table_rows                                                          11,093.0000
table_unique_wallets                                                11,093.0000
table_median_abs_lvr                        25,669,919,302,644,022,640,640.0000
table_rows_abs_lvr_below_1e4                                         2,642.0000
table_rows_abs_lvr_below_1e4_exactly_zero                            2,642.0000
dtype: float64

The column has a median absolute value of ≈ 2.6·10²², which is not a plausible USD amount, and the rows with |value| < 10⁴ are all exactly zero, so a cohort comparison restricted to small values compares zeros with zeros. The table has one row per wallet (11,093 rows, 11,093 wallets), so it cannot identify multi-chain wallets by itself.

## Liquidity ledgers and the multi-chain cohort

In [4]:
pd.DataFrame(ledgers)

,ethereum,arbitrum
rows,260521,1304039
wallets,14510,55610
first,2021-05-05 21:41:20,2021-07-12 20:43:44
last,2024-08-28 12:22:24,2024-08-28 12:50:08
share_rows_in_window,0.1209,0.1459
wallets_in_window,2742,14393
share_negative_cum_liquidity,0.0487,0.2990


The ledgers run from 2021 to 2024, so only 12–15% of rows fall in March–June 2023, and 30% of Arbitrum rows have a negative cumulative liquidity, which is impossible for real positions. They therefore cannot be used to value positions, but they do identify which wallets are active on both chains.

In [5]:
pd.Series(
    {
        k: lp[k]
        for k in [
            "wallets_on_both_chains_all_time",
            "wallets_on_both_chains_mar_jun_2023",
            "table_rows_multi_chain",
            "table_share_multi_chain",
            "median_dwell_s_multi",
            "median_dwell_s_single",
            "p_mannwhitney_dwell",
            "controlled_ols_beta_multi_chain",
            "controlled_ols_p_multi_chain",
        ]
    }
)

wallets_on_both_chains_all_time           1,733.0000
wallets_on_both_chains_mar_jun_2023         379.0000
table_rows_multi_chain                      787.0000
table_share_multi_chain                       0.0709
median_dwell_s_multi                  1,846,272.0000
median_dwell_s_single                         0.0000
p_mannwhitney_dwell                           0.0000
controlled_ols_beta_multi_chain          30,718.9817
controlled_ols_p_multi_chain                  0.1089
dtype: float64

Multi-chain wallets have far longer dwell times (median ≈ 1.8·10⁶ s versus 0 s). Once dwell time and chain are controlled, the difference in the table's loss column is not significant (p ≈ 0.11).

**Conclusion.** The multi-chain cohort can be identified (787 of 11,093 wallets), but the loss column is not a usable measure, so the multi-chain question cannot be answered with this data. A valid test needs token amounts and pool state (ticks, swaps); `crosschain_arb.lp` is ready for that input.